# Lending Club · Credit default with tabular foundation models
### TabTune on the Lexsi platform (SDK)

| Step | What you will see |
|---|---|
| 1 | Connect, create the project, upload Lending Club loans |
| 2 | Data dashboard: default rate by interest rate, FICO, debt-to-income |
| 3 | Train several foundation models zero-shot, each with the explanations it supports |
| 4 | Model comparison dashboard from the platform's own metrics |
| 5 | Explainability: global SHAP and LIME across models, training-row influence |
| 6 | Batch scoring on held-out loans and a full evaluation dashboard |
| 7 | One applicant, explained: SHAP, LIME and influential training rows |
| 8 | Advanced builds: ensemble, PAD + Bayesian Competence Fusion distillation, leaderboard |

**Before you start:** the processed Lending Club files must be in Google Drive at `MyDrive/datasets/lc/` (`X_train_final.csv`, `y_train_final.csv`, `X_val_final.csv`, `y_val_final.csv`), and you need a Lexsi SDK access token.

## 1 · Connect

In [ ]:
# ⚙️ Install the Lexsi SDK (pinned to the version the backend was tested with)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lexsi-sdk==0.1.63"], check=True)
from importlib.metadata import version
print("lexsi-sdk", version("lexsi-sdk"))

In [ ]:
# 🔐 Authenticate (paste your Lexsi SDK access token when prompted)
from lexsi_sdk import lexsi
lexsi.login()

In [ ]:
# 🔧 Demo configuration — the only cell you should need to edit
import os
ORGANIZATION = "Lexsi Labs"
WORKSPACE    = "TabTunetesting"
PROJECT      = "TFM Demo · Lending Club"      # a fresh name avoids clashing with older test projects
COMPUTE      = "T4.small"                     # GPU queue; foundational builds cannot use CPU batch types

DATA_DIR = os.environ.get("LC_DATA_DIR", "/content/drive/MyDrive/datasets/lc")
SAMPLE_TRAIN_N, SAMPLE_TEST_N = 4000, 1500    # the backend caps a build at 10,000 rows (train + test)

# Zero-shot roster. OrionMoE / OrionMSP are Lexsi models; Limix and TabPFNv3 are research-licensed.
ROSTER = ["OrionMoE", "OrionMSP", "TabICLv2", "Limix"]
RUN_SUPPORT_SET  = True     # needs 'support_set' enabled on your plan; the helper falls back automatically
RUN_ENSEMBLE     = True
RUN_DISTILLATION = True     # PAD student + Bayesian Competence Fusion over several teachers
RUN_LEADERBOARD  = True
LIME_ITERATIONS  = 200

TABPFN_TOKEN = None         # set (or use getpass) to add "TabPFNv3" to the roster
CLASS_NAMES  = ["Fully paid", "Default"]

In [ ]:
# 🧰 Helpers: idempotent project setup, safe training, metric extraction, plotting
import time, json, re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from contextlib import contextmanager
from IPython.display import display, Markdown

BLUE, ORANGE, TEAL, GREY, DARK, RED = "#2F5BEA", "#B4541A", "#1B998B", "#9AA0A8", "#16181D", "#D7263D"
pd.set_option("display.max_columns", 50)


def get_or_create(workspace, name, project_type):
    """Reuse a project if it exists, otherwise create it. Safe to rerun."""
    try:
        project = workspace.project(name)
        print(f"reusing project: {name}")
    except Exception:
        project = workspace.create_project(name, modality="tabular", project_type=project_type)
        print(f"created project: {name}")
    return project


def upload_once(project, csv_path, tag, config=None):
    """Upload a file unless this tag already has it; set the config only on the first upload."""
    try:
        files = project.files()
        existing = set(files["file_name"]) if "file_name" in files else set()
    except Exception:
        existing = set()
    name = csv_path.split("/")[-1]
    if name in existing:
        print(f"already uploaded: {name} ({tag})")
        return
    has_config = project.config() != "Not Found"
    if config is not None and not has_config:
        print(project.upload_data(csv_path, tag, config=config))
    else:
        print(project.upload_data(csv_path, tag))


@contextmanager
def tolerant_drop():
    """project.models() drops columns that some backend builds do not return."""
    orig = pd.DataFrame.drop

    def safe(self, *a, **k):
        k.setdefault("errors", "ignore")
        return orig(self, *a, **k)

    pd.DataFrame.drop = safe
    try:
        yield
    finally:
        pd.DataFrame.drop = orig


RUNS = []          # one record per training call, shown in the run log


def trained_models(project):
    with tolerant_drop():
        df = project.models()
    return df


def summary_of(project, model_name):
    try:
        return project.model_summary(model_name=model_name)
    except Exception as exc:
        print(f"no summary for {model_name}: {exc}")
        return None


def gfi(summary, method):
    """Global feature importance dict {feature: value} for 'shap' or 'lime', or {}."""
    if summary is None:
        return {}
    block = (summary.model_results or {}).get("GFI", {}) or {}
    raw = block.get(f"{method}_gfi") or {}
    out = {}
    for k, v in raw.items():
        try:
            out[k] = float(np.mean(np.abs(list(v.values())))) if isinstance(v, dict) else float(v)
        except Exception:
            pass
    return out


def support_rows(summary):
    if summary is None:
        return pd.DataFrame()
    block = ((summary.model_results or {}).get("GFI", {}) or {}).get("support_set_attribution") or {}
    return pd.DataFrame(block.get("top_support_rows", []))


def top_bar(importance, title, n=15, color=BLUE):
    s = pd.Series(importance).abs().sort_values(ascending=False).head(n)[::-1]
    fig = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h", marker_color=color))
    fig.update_layout(template="plotly_white", height=36 * len(s) + 140, title=title,
                      xaxis_title="mean |attribution|", margin=dict(l=10, r=10, t=60, b=40))
    return fig

In [ ]:
def model_names(project):
    try:
        return set(trained_models(project)["model_name"].astype(str))
    except Exception:
        return set()


def train(project, label, **kwargs):
    """train_model with timing, a status line, and the exact name of the model it created."""
    before, t0 = model_names(project), time.time()
    try:
        project.train_model(**kwargs)
        status = "ok"
    except Exception as exc:
        status = f"failed: {str(exc)[:160]}"
        xai = (kwargs.get("data_config") or {}).get("xai_method", [])
        if "support_set" in xai and "not allowed" in str(exc):
            print(f"↺ {label}: support set is not enabled for this account; retrying with SHAP and LIME")
            kwargs["data_config"] = {**kwargs["data_config"], "xai_method": [m for m in xai if m != "support_set"]}
            return train(project, label, **kwargs)
    new = sorted(model_names(project) - before)
    RUNS.append({"run": label, "strategy": kwargs.get("training_strategy"), "model": str(kwargs.get("model")),
                 "created": ", ".join(new), "status": status, "minutes": round((time.time() - t0) / 60, 1)})
    print(("✅ " if status == "ok" else "❌ ") + f"{label} → {', '.join(new) or '-'}: {status} ({RUNS[-1]['minutes']} min)")
    return new[-1] if (status == "ok" and new) else None


def base_data_config(xai):
    return {"tags": ["Training"], "test_tags": ["Test"], "feature_exclude": FEATURE_EXCLUDE,
            "xai_method": xai, "lime_explainability_iterations": LIME_ITERATIONS}


def model_config_for(models):
    models = [models] if isinstance(models, str) else list(models)
    return {"tabpfn_token": TABPFN_TOKEN} if "TabPFNv3" in models and TABPFN_TOKEN else None


def eval_metrics(summary):
    return dict((summary.model_results or {}).get("evalution_metrics", {}) or {}) if summary else {}


def modelling_info(summary):
    return dict((summary.model_results or {}).get("modelling_info", {}) or {}) if summary else {}


def features_used(summary):
    return list((summary.model_results or {}).get("feature_actual_used", []) or []) if summary else []


def ensure_inference(project, model_name):
    """Make `model_name` the active model and enable it for inference."""
    project.activate_model(model_name)
    try:
        project.update_inference_model_status(model_name, True)
    except Exception:
        pass


def case_attributions(case, kind="shap", positive="1"):
    """Signed per-feature attribution dict from a case, picking the positive class when nested."""
    raw = getattr(case, f"{kind}_feature_importance", {}) or {}
    if raw and isinstance(next(iter(raw.values())), dict):
        key = positive if positive in raw else list(raw)[-1]
        raw = raw[key]
    out = {}
    for k, v in raw.items():
        try:
            out[k] = float(v)
        except Exception:
            pass
    return out


def signed_bar(attr, title, n=12):
    s = pd.Series(attr)
    s = s.reindex(s.abs().sort_values(ascending=False).index).head(n)[::-1]
    fig = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h",
                           marker_color=[RED if v > 0 else BLUE for v in s.values]))
    fig.update_layout(template="plotly_white", height=34 * len(s) + 150, title=title,
                      xaxis_title="contribution (red pushes the prediction up)", margin=dict(l=10, r=10, t=60, b=40))
    return fig

In [ ]:
# 📐 Read the platform's inference output and score it locally
from sklearn import metrics as skm


def prediction_columns(result, uploaded):
    """Columns the platform added to the uploaded tag (prediction, probabilities, ...)."""
    return [c for c in result.columns if c not in set(uploaded.columns)]


def positive_proba(result, added, positive="1"):
    """Best guess at the positive-class probability column among the added columns."""
    numeric = [c for c in added if pd.api.types.is_numeric_dtype(result[c])
               and result[c].between(0, 1).all() and result[c].nunique() > 2]
    ranked = sorted(numeric, key=lambda c: (str(positive) not in c.split("_")[-1] and not c.endswith(str(positive)),
                                            "prob" not in c.lower() and "score" not in c.lower()))
    return ranked[0] if ranked else None


def predicted_label(result, added):
    cands = [c for c in added if result[c].nunique() <= 20]
    named = [c for c in cands if "pred" in c.lower()]
    return (named or cands or [None])[0]

In [ ]:
def classification_dashboard(y_true, p, title, threshold=0.5):
    """ROC, precision-recall, calibration, score distribution, gains and confusion matrix."""
    y = np.asarray(y_true).astype(int)
    fpr, tpr, _ = skm.roc_curve(y, p)
    prec, rec, _ = skm.precision_recall_curve(y, p)
    auc, ap = skm.roc_auc_score(y, p), skm.average_precision_score(y, p)
    bins = pd.qcut(p, 10, labels=False, duplicates="drop")
    cal = pd.DataFrame({"p": p, "y": y, "bin": bins}).groupby("bin").mean()
    order = np.argsort(-p)
    gains = np.cumsum(y[order]) / y.sum()
    share = np.arange(1, len(y) + 1) / len(y)
    pred = (p >= threshold).astype(int)
    cm = skm.confusion_matrix(y, pred)
    fig = make_subplots(rows=2, cols=3, vertical_spacing=0.16, horizontal_spacing=0.08,
                        subplot_titles=(f"ROC · AUC {auc:.3f}", f"Precision–recall · AP {ap:.3f}", "Calibration",
                                        "Score distribution by outcome", "Cumulative gains",
                                        f"Confusion matrix @ {threshold:.2f}"))
    fig.add_trace(go.Scatter(x=fpr, y=tpr, line=dict(color=BLUE, width=3), name="ROC"), 1, 1)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(color=GREY, dash="dot"), showlegend=False), 1, 1)
    fig.add_trace(go.Scatter(x=rec, y=prec, line=dict(color=ORANGE, width=3), name="PR"), 1, 2)
    fig.add_hline(y=y.mean(), line_dash="dot", line_color=GREY, row=1, col=2)
    fig.add_trace(go.Scatter(x=cal.p, y=cal.y, mode="lines+markers", line=dict(color=TEAL, width=3), name="observed"), 1, 3)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(color=GREY, dash="dot"), showlegend=False), 1, 3)
    for cls, col in ((0, BLUE), (1, RED)):
        fig.add_trace(go.Histogram(x=p[y == cls], nbinsx=30, opacity=0.6, marker_color=col,
                                   name=CLASS_NAMES[cls], histnorm="probability"), 2, 1)
    fig.add_trace(go.Scatter(x=share, y=gains, line=dict(color=BLUE, width=3), name="model"), 2, 2)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(color=GREY, dash="dot"), name="random"), 2, 2)
    fig.add_trace(go.Heatmap(z=cm[::-1], x=[f"pred {c}" for c in CLASS_NAMES], y=[f"true {c}" for c in CLASS_NAMES[::-1]],
                             text=cm[::-1], texttemplate="%{text}", colorscale="Blues", showscale=False), 2, 3)
    fig.update_layout(template="plotly_white", height=760, title=title, barmode="overlay",
                      legend=dict(orientation="h", y=-0.08))
    fig.update_xaxes(title_text="false positive rate", row=1, col=1); fig.update_yaxes(title_text="true positive rate", row=1, col=1)
    fig.update_xaxes(title_text="recall", row=1, col=2); fig.update_xaxes(title_text="mean predicted", row=1, col=3)
    fig.update_xaxes(title_text="share of applicants reviewed", row=2, col=2)
    fig.show()
    return {"roc_auc": auc, "avg_precision": ap, "brier": skm.brier_score_loss(y, p),
            "f1@thr": skm.f1_score(y, pred), "recall@thr": skm.recall_score(y, pred),
            "precision@thr": skm.precision_score(y, pred, zero_division=0)}

In [ ]:
def preflight_high_cardinality(df, cols, limit=0.95, digits=(4, 3, 2)):
    """The backend drops any column with more than 95% unique values (model.py:494-500), which also
    removes continuous features. Round those columns to fewer significant digits so they survive."""
    report = []
    for c in cols:
        if not pd.api.types.is_numeric_dtype(df[c]):
            continue
        ratio = df[c].nunique() / len(df)
        if ratio <= limit:
            continue
        fixed = None
        for d in digits:
            rounded = df[c].map(lambda v: float(f"{v:.{d}g}") if pd.notna(v) else v)
            if rounded.nunique() / len(df) <= limit:
                fixed = d
                break
        report.append({"feature": c, "unique_ratio": round(ratio, 3),
                       "fix": f"rounded to {fixed} significant digits" if fixed else "dropped by backend"})
        if fixed:
            df[c] = df[c].map(lambda v: float(f"{v:.{fixed}g}") if pd.notna(v) else v)
    return pd.DataFrame(report)

In [ ]:
# 📚 Model catalogue as enforced by the Lexsi backend (xai/core/config.py, xai/utils/checks.py)
CATALOGUE = pd.DataFrame([
    # model,          classification, regression, support_set, shap_regression, commercial,  note
    ("OrionMoE",      True,  False, False, False, "yes",        "Lexsi mixture-of-experts, inference only"),
    ("OrionMSP",      True,  False, True,  False, "yes",        "Lexsi multi-scale sparse attention"),
    ("OrionBix",      True,  False, True,  False, "yes",        "Lexsi bi-axial attention"),
    ("OrionMSPv1.5",  True,  False, False, False, "yes",        "Lexsi"),
    ("TabICL",        True,  False, True,  False, "yes",        ""),
    ("TabICLv2",      True,  True,  False, True,  "yes",        ""),
    ("TabPFN",        True,  True,  False, True,  "unverified", ""),
    ("TabPFNv26",     True,  True,  True,  True,  "unverified", ""),
    ("TabPFNv3",      True,  True,  True,  True,  "no",         "needs a tabpfn_token"),
    ("TabDPT",        True,  True,  False, True,  "unverified", ""),
    ("Mitra",         True,  True,  False, False, "yes",        "no predict_proba"),
    ("ContextTab",    True,  True,  False, False, "no",         ""),
    ("Limix",         True,  True,  True,  True,  "no",         ""),
    ("TabFM",         True,  True,  False, False, "no",         "Google"),
    ("EXAONETabular", True,  False, False, False, "no",         "LG AI Research"),
    ("ILTM",          True,  True,  False, False, "yes",        ""),
    ("XRFM",          True,  True,  False, True,  "yes",        "trains from scratch"),
], columns=["model", "classification", "regression", "support_set", "shap_regression", "commercial", "note"]).set_index("model")


def xai_for(model, task):
    """Explainability methods the backend accepts for a simple build of `model`."""
    cap = CATALOGUE.loc[model]
    if task == "regression":
        return ["shap", "lime"] if cap.shap_regression else ["lime"]
    methods = ["shap", "lime"]
    if cap.support_set and RUN_SUPPORT_SET:
        methods.append("support_set")
    return methods

In [ ]:
# 🗂️ Organization → workspace → project
organization = lexsi.organization(ORGANIZATION)
workspace = organization.workspace(WORKSPACE)
project = get_or_create(workspace, PROJECT, "classification")
if TABPFN_TOKEN and "TabPFNv3" not in ROSTER:
    ROSTER.append("TabPFNv3")
display(CATALOGUE.loc[ROSTER].style.set_caption("Models in this demo (from the backend catalogue)"))

## 2 · Lending Club data

In [ ]:
# 📥 Load the processed Lending Club files from Drive (same preprocessing as the earlier notebook)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

rd = lambda f: pd.read_csv(os.path.join(DATA_DIR, f))
X_train = rd("X_train_final.csv").rename(columns={"Unnamed: 0": "id", "id": "info"})
y_train = rd("y_train_final.csv").rename(columns={"Unnamed: 0": "id", "y": "TARGET"})
X_val   = rd("X_val_final.csv").rename(columns={"Unnamed: 0": "id", "id": "info"})
y_val   = rd("y_val_final.csv").rename(columns={"Unnamed: 0": "id", "y": "TARGET"})

train_full = X_train.merge(y_train[["id", "TARGET"]], on="id", how="inner")
val_full   = X_val.merge(y_val[["id", "TARGET"]], on="id", how="inner")
for _df in (train_full, val_full):
    _df.drop(columns=["emp_length_< 1 year"], inplace=True, errors="ignore")
    _df["TARGET"] = _df["TARGET"].astype(int).astype(str)

bad = [c for c in train_full.columns if c != "TARGET" and (re.search(r'[<>\[\]",]', c) or c != c.strip())]
train_full.drop(columns=bad, inplace=True, errors="ignore"); val_full.drop(columns=bad, inplace=True, errors="ignore")

UID, LABEL = "id", "TARGET"
FEATURE_EXCLUDE = ["info"]                    # the original loan id: an identifier, not a feature
FEATURES = [c for c in train_full.columns if c not in (UID, LABEL, *FEATURE_EXCLUDE)]
print(f"train {train_full.shape} · validation {val_full.shape} · {len(FEATURES)} features")
print("default rate (train):", f"{(train_full[LABEL] == '1').mean():.1%}")

In [ ]:
# 📊 Data dashboard: default rate by risk driver (features are pre-standardised, so we use deciles)
def default_by_decile(df, col):
    d = pd.qcut(df[col].rank(method="first"), 10, labels=[f"D{i}" for i in range(1, 11)])
    return df.assign(decile=d).groupby("decile", observed=True)[LABEL].apply(lambda s: (s == "1").mean() * 100)

drivers = [c for c in ["int_rate", "fico_range_low", "dti", "annual_inc", "revol_util", "loan_amnt"] if c in train_full][:5]
fig = make_subplots(rows=2, cols=3, vertical_spacing=0.18,
                    subplot_titles=["Outcome mix"] + [f"Default rate by {c} decile (%)" for c in drivers])
counts = train_full[LABEL].map({"0": CLASS_NAMES[0], "1": CLASS_NAMES[1]}).value_counts()
fig.add_trace(go.Bar(x=counts.index, y=counts.values, marker_color=[BLUE, RED], text=counts.values,
                     textposition="outside", showlegend=False), 1, 1)
for i, c in enumerate(drivers):
    r = default_by_decile(train_full, c)
    fig.add_trace(go.Bar(x=r.index, y=r.values, marker_color=ORANGE, showlegend=False), 1 + (i + 1) // 3, 1 + (i + 1) % 3)
fig.update_layout(template="plotly_white", height=640, title="Lending Club · who defaults?")
fig.show()

In [ ]:
# ✂️ Stratified sample, backend pre-flight check, and CSVs for upload
from sklearn.model_selection import train_test_split

def stratified_sample(df, n, seed=42):
    if n >= len(df):
        return df.reset_index(drop=True)
    _, s = train_test_split(df, test_size=n / len(df), stratify=df[LABEL], random_state=seed)
    return s.reset_index(drop=True)

train_sample = stratified_sample(train_full, SAMPLE_TRAIN_N)
test_sample  = stratified_sample(val_full, SAMPLE_TEST_N)

both = pd.concat([train_sample, test_sample])
fixes = preflight_high_cardinality(both, FEATURES)
train_sample, test_sample = both.iloc[:len(train_sample)].copy(), both.iloc[len(train_sample):].copy()
display(fixes if len(fixes) else Markdown("No feature would be dropped by the backend's 95%-unique rule."))

TRAIN_CSV, TEST_CSV = "/content/lc_demo_train.csv", "/content/lc_demo_test.csv"
train_sample.to_csv(TRAIN_CSV, index=False)
test_sample.to_csv(TEST_CSV, index=False)
print(f"train sample {train_sample.shape} · test sample {test_sample.shape} · "
      f"default rate {(train_sample[LABEL] == '1').mean():.1%}")

In [ ]:
# ⬆️ Upload once: Training sets the project config, Test is the hold-out the platform scores against
config = {"unique_identifier": UID, "true_label": LABEL, "pred_label": "",
          "feature_exclude": FEATURE_EXCLUDE, "drop_duplicate_uid": True, "handle_errors": True}
upload_once(project, TRAIN_CSV, "Training", config=config)
upload_once(project, TEST_CSV, "Test")
display(project.files()[["file_name", "tag", "status", "created_at"]])
print("tags:", project.tags())

## 3 · Zero-shot training

Each model is trained with `tunning_strategy="inference"`: no weights change, the uploaded loans become the model's context. Every build requests only the explanations the backend accepts for that model, and is scored by the platform on the **Test** tag.

In [ ]:
# 🚀 One simple build per model
BUILDS = {}
for model in ROSTER:
    xai = xai_for(model, "classification")
    name = train(project, f"{model} · zero-shot", model=model, training_strategy="simple",
                 compute_type=COMPUTE, tunning_strategy="inference",
                 data_config=base_data_config(xai), model_config=model_config_for(model))
    if name:
        BUILDS[model] = name
print("\nbuilt:", BUILDS)

## 4 · Model comparison

In [ ]:
# 🏁 Platform metrics for every build, side by side
SUMMARIES = {m: summary_of(project, n) for m, n in BUILDS.items()}
rows = []
for m, s in SUMMARIES.items():
    em = eval_metrics(s)
    rows.append({"model": m, "build": BUILDS[m], **{k: em.get(k) for k in
                 ("RocAucScore", "F1Score", "Precision", "Recall", "Accuracy", "Mcc")},
                 "features used": len(features_used(s)), "commercial": CATALOGUE.loc[m, "commercial"]})
board = pd.DataFrame(rows).set_index("model").sort_values("RocAucScore", ascending=False)
display(board.style.format(precision=3, na_rep="-").background_gradient(subset=["RocAucScore"], cmap="Blues")
        .set_caption("Scored by the platform on the Test tag"))

long = board[["RocAucScore", "F1Score", "Recall", "Precision"]].reset_index().melt("model", var_name="metric")
fig = px.bar(long, x="metric", y="value", color="model", barmode="group", text_auto=".2f",
             color_discrete_sequence=[BLUE, ORANGE, TEAL, GREY, RED, DARK])
fig.update_layout(template="plotly_white", height=460, title="Zero-shot foundation models on held-out Lending Club loans",
                  yaxis_range=[0, 1])
fig.show()
BEST = board["RocAucScore"].astype(float).fillna(-1).idxmax()
display(Markdown(f"**Best by ROC AUC: `{BEST}` ({BUILDS[BEST]})** — used for scoring and the case view below."))

## 5 · Explainability across models

* **Global SHAP and LIME** — which features drive each model overall.
* **Agreement** — do different foundation models rely on the same drivers?
* **Training-row influence (support set)** — for in-context models, which past loans in the context moved the predictions most. Only in-context learners can answer this.

In [ ]:
# 🔍 Global SHAP: top drivers per model, and how much the models agree
shap = {m: gfi(s, "shap") for m, s in SUMMARIES.items() if gfi(s, "shap")}
lime = {m: gfi(s, "lime") for m, s in SUMMARIES.items() if gfi(s, "lime")}
if shap:
    top_bar(shap.get(BEST) or next(iter(shap.values())), f"Global SHAP · {BEST}").show()
    mat = pd.DataFrame(shap).fillna(0)
    mat = mat.div(mat.max())                                  # each model scaled to its own top feature
    top = mat.mean(axis=1).sort_values(ascending=False).head(15).index
    fig = px.imshow(mat.loc[top], color_continuous_scale="Blues", aspect="auto", text_auto=".2f",
                    title="Relative SHAP importance · top 15 features across models")
    fig.update_layout(height=620, template="plotly_white"); fig.show()
    if mat.shape[1] > 1:
        display(mat.rank(ascending=False).corr(method="spearman").round(2)
                .style.background_gradient(cmap="Blues", vmin=0, vmax=1)
                .set_caption("Rank agreement of SHAP importance between models (Spearman)"))
else:
    print("No SHAP importance stored yet.")

In [ ]:
# 🔁 SHAP vs LIME for the best model
if BEST in shap and BEST in lime:
    both_ = pd.DataFrame({"SHAP": pd.Series(shap[BEST]), "LIME": pd.Series(lime[BEST])}).fillna(0)
    both_ = both_.div(both_.max())
    top = both_.max(axis=1).sort_values(ascending=False).head(12).index[::-1]
    fig = go.Figure([go.Bar(y=top, x=both_.loc[top, "SHAP"], name="SHAP", orientation="h", marker_color=BLUE),
                     go.Bar(y=top, x=both_.loc[top, "LIME"], name="LIME", orientation="h", marker_color=ORANGE)])
    fig.update_layout(template="plotly_white", barmode="group", height=560,
                      title=f"Two explanation methods, one model · {BEST} (scaled to each method's top feature)")
    fig.show()

In [ ]:
# 🧷 Training-row influence (support set): which loans in the context matter most
ss = {m: support_rows(s) for m, s in SUMMARIES.items()}
ss = {m: d for m, d in ss.items() if len(d)}
if ss:
    fig = go.Figure()
    for m, d in ss.items():
        d = d.sort_values("influence", ascending=False).head(25)
        fig.add_trace(go.Bar(x=d["support_index"].astype(str), y=d["influence"], name=m))
    fig.update_layout(template="plotly_white", barmode="group", height=420, xaxis_title="training row (context position)",
                      title="Top 25 most influential training rows · leave-one-out influence")
    fig.show()
    m0 = next(iter(ss))
    display(SUMMARIES[m0].support_attribution().head(10))
else:
    print("No support-set attribution stored (not enabled on this plan, or no capable model in the roster).")

## 6 · Batch scoring on held-out loans

In [ ]:
# 📦 Score the Test tag with the best model and evaluate it locally
ensure_inference(project, BUILDS[BEST])
scored = project.model_inference(tag="Test", model_name=BUILDS[BEST], pod=COMPUTE)
scored = scored.merge(test_sample[[UID, LABEL]].rename(columns={LABEL: "actual"}), on=UID, how="left")
p = scored["pred_proba_AutoML"].astype(float).to_numpy()      # probability of default (class 1)
y = (scored["actual"].astype(str) == "1").astype(int).to_numpy()

thr_grid = np.linspace(0.05, 0.95, 91)
best_thr = float(thr_grid[np.argmax([skm.f1_score(y, (p >= t).astype(int)) for t in thr_grid])])
local = classification_dashboard(y, p, f"{BUILDS[BEST]} · {len(y):,} held-out loans", threshold=best_thr)
display(pd.Series(local, name=f"threshold {best_thr:.2f} (max F1)").to_frame().style.format("{:.3f}"))

In [ ]:
# 💼 Business view: reviewing the riskiest loans first
dec = pd.qcut(pd.Series(p).rank(method="first", ascending=False), 10, labels=[f"{i * 10}%" for i in range(1, 11)])
gain = pd.DataFrame({"decile": dec, "y": y}).groupby("decile", observed=True)["y"].agg(["sum", "mean"])
gain["cum_capture"] = gain["sum"].cumsum() / gain["sum"].sum() * 100
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=gain.index, y=gain["mean"] * 100, name="default rate in band (%)", marker_color=ORANGE))
fig.add_trace(go.Scatter(x=gain.index, y=gain["cum_capture"], name="defaults captured (%)",
                         line=dict(color=BLUE, width=3)), secondary_y=True)
fig.update_layout(template="plotly_white", height=430, xaxis_title="riskiest share of loans reviewed",
                  title=f"Reviewing the top 20% riskiest loans captures {gain['cum_capture'].iloc[1]:.0f}% of defaults")
fig.show()

In [ ]:
# ⚖️ Group check (computed locally: the platform API does not run fairness for TFM builds)
groups = [c for c in test_sample.columns if c.startswith("home_ownership_")]
if groups:
    g = test_sample.set_index(UID)[groups].idxmax(axis=1).str.replace("home_ownership_", "")
    view = scored.assign(group=scored[UID].map(g), flagged=(p >= best_thr).astype(int), y=y)
    rates = view.groupby("group").agg(loans=("y", "size"), default_rate=("y", "mean"), flag_rate=("flagged", "mean"),
                                      recall=("flagged", lambda s: s[view.loc[s.index, "y"] == 1].mean()))
    display(rates.style.format({"default_rate": "{:.1%}", "flag_rate": "{:.1%}", "recall": "{:.1%}"})
            .set_caption("Flag rate and recall of defaults by home ownership"))

## 7 · One applicant, explained

In [ ]:
# 🧑‍💼 Pick the riskiest applicant in the Test tag who actually defaulted, and explain the decision
CASE_UID = str(scored.loc[scored["actual"].astype(str) == "1"].sort_values("pred_proba_AutoML", ascending=False)[UID].iloc[0])
case_xai = ["shap", "lime"] + (["support_set"] if CATALOGUE.loc[BEST, "support_set"] and RUN_SUPPORT_SET else [])
try:
    case = project.case_predict(unique_identifier=CASE_UID, tag="Test", model_name=BUILDS[BEST], xai=case_xai)
except Exception as exc:
    print("support set not available for this case, retrying with SHAP and LIME:", str(exc)[:100])
    case = project.case_predict(unique_identifier=CASE_UID, tag="Test", model_name=BUILDS[BEST], xai=["shap", "lime"])
display(case.inference_output())

In [ ]:
# 🧾 Case dashboard: why this applicant was scored as high risk
sh, li = case_attributions(case, "shap"), case_attributions(case, "lime")
if sh:
    signed_bar(sh, f"SHAP · applicant {CASE_UID} · red pushes towards default").show()
if li:
    signed_bar(li, f"LIME · applicant {CASE_UID}").show()
top = pd.Series(sh or li).abs().sort_values(ascending=False).head(6).index
row = test_sample.assign(_k=test_sample[UID].astype(str)).set_index("_k").loc[CASE_UID]
display(pd.DataFrame({"value (standardised)": row[top],
                      "train average": train_sample[top].mean()}).round(2)
        .style.set_caption("The applicant's top drivers next to the portfolio average"))
ss_case = case.xai_support_set()
if isinstance(ss_case, pd.DataFrame):
    display(ss_case.head(10).style.set_caption("Training rows that moved this prediction most (leave-one-out)"))

## 8 · Advanced builds

* **Ensemble** — greedy selection over the zero-shot models (ensembles are explained with LIME).
* **PAD + Bayesian Competence Fusion** — several foundation-model teachers fused label-free into one fast PAD student (`MultiTeacher_distilled_vN`).
* **Leaderboard** — the platform races the models on an internal hold-out and refits the winner.

In [ ]:
# 🤝 Ensemble · 🧪 PAD + BCF distillation · 🏆 Leaderboard
ADV = {}
pool = list(BUILDS)[:3]
if RUN_ENSEMBLE and len(pool) >= 2:
    ADV["Ensemble"] = train(project, "Ensemble · greedy", model=pool, training_strategy="ensemble",
                            compute_type=COMPUTE, tunning_strategy="inference",
                            data_config=base_data_config(["lime"]), model_config=model_config_for(pool),
                            ensemble_config={"ensemble_strategy": "greedy_selection", "metric": "log_loss",
                                             "weight_scheme": "performance", "cv_folds": 5,
                                             "holdout_fraction": 0.2, "greedy_ensemble_size": 50})
if RUN_DISTILLATION and len(pool) >= 2:
    ADV["PAD + BCF"] = train(project, "Distillation · PAD student, BCF fusion", model=pool,
                             training_strategy="distillation", compute_type=COMPUTE,
                             data_config=base_data_config(["shap", "lime"]), model_config=model_config_for(pool),
                             distillation_config={"student": "pad_gbdt", "gbdt_backend": "lgbm",
                                                  "temperature": 3.0, "alpha": 0.7, "n_folds": 5,
                                                  "adaptive_temperature": True, "confidence_weighting": True,
                                                  "bcf_pool_sharpness": 3.0, "bcf_adaptive_kappa": True,
                                                  "bcf_correlation_correction": True})
if RUN_LEADERBOARD and len(BUILDS) >= 2:
    ADV["Leaderboard"] = train(project, "Leaderboard · ROC AUC", model=list(BUILDS), training_strategy="leaderboard",
                               compute_type=COMPUTE, tunning_strategy="inference",
                               data_config=base_data_config(["shap", "lime"]), model_config=model_config_for(list(BUILDS)),
                               leaderboard_config={"rank_by": "roc_auc_score"})
ADV = {k: v for k, v in ADV.items() if v}
print(ADV)

In [ ]:
# 📊 Everything built today, on the same Test tag
allb = {**{m: BUILDS[m] for m in BUILDS}, **ADV}
rows = []
for label, name in allb.items():
    s = summary_of(project, name)
    em = eval_metrics(s)
    rows.append({"build": label, "model": name, "RocAucScore": em.get("RocAucScore"), "F1Score": em.get("F1Score"),
                 "Recall": em.get("Recall"), "Precision": em.get("Precision")})
    mi = modelling_info(s)
    if label == "Leaderboard" and mi.get("leaderboard"):
        display(Markdown(f"**Leaderboard winner: `{mi.get('winner_model_name')}`**"))
        display(pd.DataFrame(mi["leaderboard"]))
    if label == "PAD + BCF" and mi.get("bcf_diagnostics"):
        display(Markdown("**BCF diagnostics** (teacher reliability learned without labels)"))
        display(pd.json_normalize(mi["bcf_diagnostics"]).T)
final = pd.DataFrame(rows).set_index("build")
display(final.style.format(precision=3, na_rep="-").background_gradient(subset=["RocAucScore"], cmap="Blues"))
fig = px.bar(final.reset_index(), x="build", y="RocAucScore", color="build", text_auto=".3f",
             title="ROC AUC on held-out loans · zero-shot models vs advanced builds")
fig.update_layout(template="plotly_white", height=430, showlegend=False, yaxis_range=[0.5, 1]); fig.show()

In [ ]:
# 🧾 Run log for this session
display(pd.DataFrame(RUNS))
with tolerant_drop():
    display(project.models()[["model_name", "status", "tabular_model_type", "instance_type", "created_at"]]
            .sort_values("created_at", ascending=False).head(20))

## Summary

| Capability | Where it appeared |
|---|---|
| Zero-shot foundation models, no training | section 3 |
| Platform-scored metrics on a hold-out tag | section 4 |
| Global SHAP and LIME, cross-model agreement | section 5 |
| Training-row influence (support set), unique to in-context models | sections 5 and 7 |
| Batch scoring, ROC / PR / calibration / gains | section 6 |
| Per-applicant explanation | section 7 |
| Ensemble, PAD + BCF multi-teacher distillation, leaderboard | section 8 |

**Notes for the presenter**
* Features in this extract are pre-standardised, so explanations read in standard-deviation units.
* Support-set attribution needs `support_set` on the account plan; without it the notebook falls back to SHAP and LIME automatically.
* SHAP is computed for binary targets; ensembles are explained with LIME.
* The group check in section 6 is computed in the notebook from the platform's predictions.